# Chia Flair theo mask kernel5_non_tumor (bán kính = 5)
Notebook này sẽ đọc hai file NIfTI:
- Flair: dataset/BraTS2021_Training_Data/BraTS2021_00002/BraTS2021_00002_flair.nii.gz
- Mask kernel5_non_tumor: dataset/BraTS2021_Training_Data/BraTS2021_00002/BraTS2021_00002_kernel5_non_tumor.nii.gz
Với mỗi voxel có giá trị 1 trong mask, sẽ cắt một khối con từ Flair, tâm tại voxel đó và bán kính 5 (kích thước khối 11x11x11), sau đó lưu từng khối dưới dạng .nii.gz.

In [1]:
# Đọc NIfTI và cắt khối con theo mask
import numpy as np
import nibabel as nib
from pathlib import Path
import time

def split(fileId: str, radius: int = 5) -> None:
  flair_path = Path(f"dataset/BraTS2021_Training_Data/{fileId}/{fileId}_flair.nii.gz")
  mask_path = Path(f"dataset/BraTS2021_Training_Data/{fileId}/{fileId}_kernel5_non_tumor.nii.gz")
  output_dir = Path(f"dataset/cnn/k5_r5/non_tumor")
  output_dir.mkdir(parents=True, exist_ok=True)

  # Kiểm tra tồn tại file
  assert flair_path.exists(), f"Không tìm thấy Flair tại: {flair_path}"
  assert mask_path.exists(), f"Không tìm thấy Mask tại: {mask_path}"

  t0 = time.time()
  flair_img = nib.load(str(flair_path))
  mask_img = nib.load(str(mask_path))

  flair = flair_img.get_fdata()
  mask = mask_img.get_fdata()

  # Kiểm tra kích thước
  assert flair.shape[:3] == mask.shape[:3], f"Shape khác nhau: flair={flair.shape}, mask={mask.shape}"
  shape = flair.shape[:3]

  # Lấy các toạ độ có giá trị 1 trong mask (dùng isclose để tránh sai số kiểu float)
  coords = np.argwhere(np.isclose(mask, 1.0))
  print("Số lượng tâm (value=1) trong mask:", coords.shape[0])

  A = flair_img.affine.copy()
  count = 0

  for (i, j, k) in coords:
      # Xác định phạm vi cắt (bao gồm cả biên, clamp trong [0, dim])
      r = radius
      start_i = max(0, int(i - r))
      start_j = max(0, int(j - r))
      start_k = max(0, int(k - r))
      end_i_ex = min(shape[0], int(i + r + 1))  # end exclusive
      end_j_ex = min(shape[1], int(j + r + 1))
      end_k_ex = min(shape[2], int(k + r + 1))

      patch = flair[start_i:end_i_ex, start_j:end_j_ex, start_k:end_k_ex]

      # Tạo affine mới: gốc (0,0,0) của patch ứng với (start_i, start_j, start_k) trong ảnh gốc
      new_affine = A.copy()
      offset_vox = np.array([start_i, start_j, start_k], dtype=float)
      new_affine[:3, 3] = (A[:3, :3] @ offset_vox) + A[:3, 3]

      # Lưu file NIfTI
      base = flair_path.stem  # ví dụ: BraTS2021_00002_flair
      fname = output_dir / f"{base}_chunk_r{r}_i{i}_j{j}_k{k}.nii.gz"
      nib.save(nib.Nifti1Image(patch, new_affine, flair_img.header), str(fname))
      count += 1

  print(f"{fileId} hoàn tất. Tổng số khối đã lưu: {count} - Thời gian: {time.time() - t0:.2f}s")

In [2]:
import os

datasetDir = os.path.join('./dataset/BraTS2021_Training_Data')

listDir = os.listdir(datasetDir)
listDir.sort()
listDir
# Iterate over files in directory
listDir = os.listdir(datasetDir)
listDir.sort()
for name in listDir:
  if name.startswith('BraTS2021_'):
    split(name, radius=5)

Số lượng tâm (value=1) trong mask: 913
BraTS2021_00002 hoàn tất. Tổng số khối đã lưu: 913 - Thời gian: 3.65s
Số lượng tâm (value=1) trong mask: 1142
BraTS2021_00003 hoàn tất. Tổng số khối đã lưu: 1142 - Thời gian: 4.37s
Số lượng tâm (value=1) trong mask: 866
BraTS2021_00005 hoàn tất. Tổng số khối đã lưu: 866 - Thời gian: 3.29s
Số lượng tâm (value=1) trong mask: 1027
BraTS2021_00006 hoàn tất. Tổng số khối đã lưu: 1027 - Thời gian: 3.93s
Số lượng tâm (value=1) trong mask: 1057
BraTS2021_00008 hoàn tất. Tổng số khối đã lưu: 1057 - Thời gian: 4.03s
Số lượng tâm (value=1) trong mask: 1053
BraTS2021_00009 hoàn tất. Tổng số khối đã lưu: 1053 - Thời gian: 4.02s
Số lượng tâm (value=1) trong mask: 1025
BraTS2021_00011 hoàn tất. Tổng số khối đã lưu: 1025 - Thời gian: 5.14s
Số lượng tâm (value=1) trong mask: 1029
BraTS2021_00012 hoàn tất. Tổng số khối đã lưu: 1029 - Thời gian: 6.33s
Số lượng tâm (value=1) trong mask: 1027
BraTS2021_00014 hoàn tất. Tổng số khối đã lưu: 1027 - Thời gian: 4.15s
Số lư